In [ ]:
import os
import math
import time
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from functools import partial
from types import SimpleNamespace
from omegaconf import OmegaConf, open_dict

import torch
from jepa import JEPA
from module import ARPredictor, Embedder, MLP, SIGReg

from train import lejepa_forward
from train_no_hydra import build_loaders, build_model

In [ ]:
CONFIG_DIR = Path(os.path.abspath('./config/train/'))
print(CONFIG_DIR)

def to_device(batch, device):
    return {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}

def load_cfg(data: str = "pusht"):
    cfg = OmegaConf.load(CONFIG_DIR / "Fast-lewm.yaml")
    cfg.pop("defaults")
    cfg.data = OmegaConf.load(CONFIG_DIR / "data" / f"{data}.yaml")
    cfg.subdir = datetime.now().strftime("%Y%m%d-%H%M%S")  # replaces ${hydra:job.id}

    # --- knobs ---------------------------------------------------------------
    # RAM in flight ~= num_workers * prefetch_factor * batch_size
    #                  * (max_transitions * action_frame_skip + 1) frames * 602 KB
    cfg.loader.batch_size = 16
    cfg.num_workers = 6
    cfg.loader.prefetch_factor = 3
    cfg.wm.action_frame_skip = 5
    cfg.wm.dense_supervision.max_transitions = 5
    cfg.wandb.enabled = True
    return cfg

In [ ]:
cfg = load_cfg('pusht')
run_dir = os.path.abspath('./results/' + cfg.subdir)
print(run_dir)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train, val = build_loaders(cfg)

In [ ]:
model = build_model(cfg).to(device)

In [ ]:
sigreg = SIGReg(**cfg.loss.sigreg.kwargs).to(device)

In [ ]:
opt = torch.optim.AdamW(
    model.parameters(),
    lr=float(cfg.optimizer.lr),
    weight_decay=float(cfg.optimizer.weight_decay),
)
total_steps = cfg.trainer.max_epochs * len(train)
warmup_steps = max(1, int(0.01 * total_steps))

print(total_steps, warmup_steps)

In [ ]:
from stable_pretraining.data.dataset_stats import ImageNet

def lr_factor(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

def inverse_transform(pixels):
    """Undo utils.get_img_preprocessor: normalized float (..., 3, H, W) -> uint8 (..., H, W, 3).

    Forward was: uint8 / 255 -> (x - ImageNet mean) / ImageNet std -> Resize(img_size).
    Exact when img_size equals the stored frame size (224 for pusht: Resize is a no-op);
    otherwise you get the resized frame back.
    """
    mean = pixels.new_tensor(ImageNet["mean"]).view(3, 1, 1)
    std = pixels.new_tensor(ImageNet["std"]).view(3, 1, 1)
    x = (pixels.float() * std + mean) * 255
    return x.round().clamp(0, 255).to(torch.uint8).movedim(-3, -1)

In [ ]:
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)

In [ ]:
logs = {}
shim = SimpleNamespace(model=model, sigreg=sigreg, log_dict=lambda d, **_: logs.update(d))

In [ ]:
cfg.trainer.max_epochs

In [ ]:
model.train()
print()

In [ ]:
for step, batch in enumerate(train):
    print(step)
    break

In [ ]:
orig_pixels = inverse_transform(batch['pixels']).numpy()
orig_pixels.shape

In [ ]:
from IPython.display import display
from PIL.Image import fromarray
from train_plain2 import jepa_losses

In [ ]:
video_tracklet = orig_pixels[0]
video_tracklet.dtype

In [ ]:
video_tracklet.shape

In [ ]:
# for img in video_tracklet:
#     display(fromarray(img))

In [ ]:
batch = to_device(batch, device)

In [ ]:
batch.keys()

In [ ]:
for k,v in batch.items():
    print(f'{k}: {v.shape}')

In [ ]:
with torch.autocast(device.type, dtype=torch.bfloat16):
    pred = jepa_losses(model, sigreg, batch, cfg)

In [ ]:
pred["loss"].backward()

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.trainer.gradient_clip_val)

In [ ]:
opt.step()
sched.step()
opt.zero_grad(set_to_none=True)

In [ ]:
pred.keys()

In [1]:
import pybullet as p

ModuleNotFoundError: No module named 'pybullet'